# Colab map test

A small notebook to find out **which kind of map fails in Google Colab**. Run the cells one by one and
note which test shows a map and which shows an empty cell.

| Test | What it shows | If it stays empty |
|---|---|---|
| 1. folium | a plain HTML map | Colab or the browser blocks map tiles |
| 2. ipyleaflet | an interactive widget map | Colab widget support is not working |
| 3. sitex site map | the map used to draw the site | the problem is in `sitex`, not in Colab |
| 4. Click check | reads back two clicks | clicks do not reach Python |
| 5. leafmap picker (optional) | the map used in `03-NA04` and `03-NA05` | leafmap widgets do not load |

## 0. Install and check versions

In [ ]:
%pip install -q "sitex[data] @ git+https://github.com/ArchiColab/sitex.git"

In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version.split()[0])
print("In Colab:", "google.colab" in sys.modules)
for pkg in ["ipywidgets", "ipyleaflet", "leafmap", "folium", "sitex"]:
    try:
        print(f"{pkg:11s}", md.version(pkg))
    except md.PackageNotFoundError:
        print(f"{pkg:11s}", "not installed")

try:
    import sitex.core.colab
    print("sitex has the Colab widget helper: yes")
except ImportError:
    print("sitex has the Colab widget helper: NO")
    print("-> GitHub does not have the fix yet, or the runtime still holds the old install.")
    print("   Push sitex, then Runtime > Disconnect and delete runtime, and start again.")

## 1. folium (plain HTML map)

**Expected:** a street map of Pleiku.

In [ ]:
import folium

folium.Map(location=[13.98, 108.0], zoom_start=14)

## 2. ipyleaflet (interactive widget)

**Expected:** a street map you can drag. An empty cell or a line starting with `Map(` means the widget did not render.

In [ ]:
from google.colab import output

output.enable_custom_widget_manager()

from ipyleaflet import Map

Map(center=[13.98, 108.0], zoom=14)

## 3. sitex site map

**Expected:** a satellite map with a yellow box. This is the map from `01-SITE-3D_Model`.

In [ ]:
from IPython.display import display
from sitex.data import aoi

site_map, default_bbox = aoi.build_aoi_map(13.98, 108.0, dist_m=1000, zoom=16)
display(site_map)

## 4. Click check

Click two opposite corners on the map above, then run this cell.

**Expected:** a bounding box that differs from the default box. If it prints the default box, the clicks do not reach Python.

In [ ]:
site = aoi.resolve_aoi_from_map(site_map, default_bbox, slug="test")
print("Default box:", tuple(round(v, 5) for v in default_bbox))
print("Your box   :", tuple(round(v, 5) for v in site.bbox))

## 5. leafmap picker (optional)

The map used to place a new facility in `03-NA04` and `03-NA05`. Needs the `network` extras, which take a few minutes to install.

In [ ]:
%pip install -q "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

In [ ]:
from shapely.geometry import box
from sitex.network import poi_accessibility as pa

picker = pa.build_picker_map(box(107.99, 13.97, 108.04, 14.0))
display(picker)

## What to send back

For every test: did a map show up (yes or no)? For any empty cell, press **F12** in the browser,
open the **Console** tab, and copy the first red line.